# Multi-Agent LLM Discussion for Paper Reviewing

**Workshop — 30 to 60 minutes**

In this workshop you will build a multi-agent peer-review system using the [SynDisco](https://github.com/dimits-ts/syndisco) library. By the end you will have a working Python script that simulates structured academic peer review through multiple LLM agents - and you will have seen why that produces better critiques than a single prompt.

## Objectives

By the end of this session you will be able to:

1. **Design** a role-based multi-agent discussion system to solve a specific problem
2. **Implement** a multi-round interaction loop using Python
3. **Compare** single-prompt vs multi-agent approaches
4. **Justify** for what tasks synthetic discussion is appropriate

---

## Prerequisites

- Basic Python
- Basic familiarity with LLMs

---
```
This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see <http://www.gnu.org/licenses/>.

You may contact the author at dim.tsirmpas@aueb.gr
```
---


In [1]:
%set_env CUDA_VISIBLE_DEVICES=6

env: CUDA_VISIBLE_DEVICES=6


%pip install syndisco --quiet

In [2]:
import syndisco
from syndisco import TransformersModel

model = TransformersModel(
    model_path="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    name="base_model",
    max_out_tokens=1500,
)
syndisco.logging_setup(
    print_to_terminal=True,
    write_to_file=False,
    level="warning",
    use_colors=True
)


## 1. What are we doing here?

### What are synthetic discussions?


### Why are we bothering with synthetic discussions?

### What problems can I solve with synthetic discussions?



## 2. Solving a real task: Reviewing a paper

Before building the multi-agent system, we should consider simpler approaches. Why not use simple prompting?

Below is a short excerpt of a niche NLP paper. Let's try to review it. If you are really curious, why not replace this with your own paper?

In [3]:
# --- Paper abstract to review -----------------------------------------------
ABSTRACT = """
Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation tasks show these
models to be superior in quality while being more parallelizable and requiring
significantly less time to train.
""".strip()

Prompting is fairly simple. We just need to define a system prompt (to clue our model in what the task is), and a user prompt (which contains the actual "meat" of our request).

In [4]:
BASELINE_SYSTEM = "You are an expert academic reviewer. Review the paper excerpt provided."
BASELINE_USER   = f"Please review the following paper abstract:\n\n{ABSTRACT}"

baseline_review = model.prompt(BASELINE_SYSTEM, BASELINE_USER)

print("=" * 60)
print("BASELINE — single-prompt review")
print("=" * 60)
print(baseline_review)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASELINE — single-prompt review
### Review of the Paper Abstract: "Attention Is All You Need"

#### Strengths:
1. **Innovative Approach**: The abstract clearly highlights the novelty of the proposed model, the Transformer, which is based solely on attention mechanisms. This sets it apart from existing models that rely on recurrent or convolutional neural networks.
2. **Simplicity Emphasis**: The phrase "new simple network architecture" emphasizes the simplicity of the proposed model, which is a significant selling point in the field of deep learning.
3. **Performance Claims**: The abstract mentions that the Transformer outperforms existing models in terms of quality, which is a strong claim that would attract readers interested in performance improvements.
4. **Parallelizability and Training Time**: The abstract also points out the benefits of the model in terms of parallelizability and reduced training time, which are important practical considerations for large-scale applications.

#

There is of course no automated way of evaluating this review. What we can do is have a look at the output ourselves, and determine some task-specific quality checks. Since our task is academic reviewing these checks could look something like this:

---

* [ ] **On track?**
  Does the response actually address the question?

* [ ] **Clear + readable?**
  Is it easy to follow, or a bit messy?

* [ ] **Some real thinking?**
  Does it go beyond obvious or generic points?

* [ ] **Balanced?**
  Does it consider more than one angle, or just present a single take?

* [ ] **Any pushback?**
  Does it question assumptions or just accept them?

* [ ] **Feels complete?**
  Do you feel like you got a useful answer, or something partial/shallow?

* [ ] **Feels realistic?**
  Does this read like an actual human-generated review? Could you discern between the two?

* [ ] **Overall feel**
  Does this feel genuinely helpful for your task?

*(We'll come back to this exact checklist later)*

---

## 3. Adding spice to the reviewer

A simple way of generating more diverse and targeted answers is by adding more details to the system prompt. While this can be trivially achieved by directly changing the system prompt above (try it out!), we can use a standard template to make our life easier when we define multiple LLM paricipants.

Our library already takes care of basic templating using the `Actor` class. Let's define a very skeptical reviewer using the following attributes:
- **`persona`** — a dict of attributes that shape its voice (role, expertise, style)
- **`context`** — what the discussion is about (ideally shared across all agents)
- **`instructions`** — same as before, bbut can be customized for each agent

In [5]:
from syndisco import Actor


REVIEW_CONTEXT = (
    f"You are participating in a structured academic peer-review discussion. "
    f"The paper under review is:\n\n{ABSTRACT}"
)

REVIEW_INSTRUCTIONS = "Write a full, professional review of the article for a prestigious journal."

skeptical_reviewer = Actor(
    model=model,
    persona={
        "role": "Skeptical Reviewer",
        "expertise": "critical analysis, assumption-testing, identifying gaps",
        "style": "challenging, rigorous, devil's advocate",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="SkepticalReviewer",
)

skeptical_review = skeptical_reviewer.speak()
print("=" * 60)
print("SKEPTICAL — single-prompt review")
print("=" * 60)
print(skeptical_review)

SKEPTICAL — single-prompt review
**Review by SkepticalReviewer**

**Title:** Attention Is All You Need

**Abstract Summary:**
The authors present a novel approach to sequence transduction models, proposing a Transformer architecture that relies solely on attention mechanisms, eliminating the need for recurrent or convolutional layers. They claim this model outperforms existing methods in terms of quality, parallelizability, and training time.

**Strengths:**
1. **Innovative Approach:** The complete reliance on attention mechanisms is a significant departure from traditional architectures, which could lead to new insights into the nature of sequence processing.
2. **Performance Claims:** The reported performance improvements on machine translation tasks are compelling, especially if they hold up under rigorous testing.
3. **Parallelizability:** The ability to train the model more efficiently could have substantial practical implications for large-scale applications.

**Weaknesses and Ar

What have we gained by creating a different age? Generally speaking, we have a whole other Point of View, which we would have otherwised missed. This is the core motivation of using different agents.

Let's spice things up even further. We are using a small model, so our inference costs are minimal. Why not simulate the entire first round of the review?

## 4. Multi-agent prompting

Lets define four reviewers total. We already have a skeptical reviewer, so think about what other *points of view* would be beneficial for the review.

We should also add a **meta-reviewer** as a participant here — they should join at the end of the review cycle, when all other reviews have been posted.

In [6]:
# --- Methodology Reviewer ---------------------------------------------------
methodology_reviewer = Actor(
    model=model,
    persona={
        "role": "Methodology Reviewer",
        "expertise": "experimental design, statistics, reproducibility",
        "style": "precise, evidence-focused, constructive",
    },
    context=REVIEW_CONTEXT, # Shared context injected into every agent's system prompt
    instructions=REVIEW_INSTRUCTIONS,
    name="MethodologyReviewer",
)

# --- Domain Reviewer --------------------------------------------------------
domain_reviewer = Actor(
    model=model,
    persona={
        "role": "Domain Reviewer",
        "expertise": "natural language processing, deep learning, related literature",
        "style": "scholarly, literature-aware, contextual",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="DomainReviewer",
)

# --- Meta-Reviewer ----------------------------------------------------------
# Joins only in round 3; SynDisco's turn order will handle this.
meta_reviewer = Actor(
    model=model,
    persona={
        "role": "Meta-Reviewer",
        "expertise": "synthesis, editorial judgment, final recommendations",
        "style": "balanced, decisive, structured",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "You have read all previous reviewer comments. Synthesise the discussion "
        "into a final structured judgment. Your output must contain:\n"
        "  • Summary of main strengths\n"
        "  • Summary of main weaknesses\n"
        "  • Overall recommendation (Accept / Major Revision / Reject) with a one-sentence rationale."
    ),
    name="MetaReviewer",
)

print("Agents created:")
for agent in [methodology_reviewer, domain_reviewer, skeptical_reviewer, meta_reviewer]:
    print(f"  • {agent.get_actor_name()}")

Agents created:
  • MethodologyReviewer
  • DomainReviewer
  • SkepticalReviewer
  • MetaReviewer


In round 1 each of the three reviewers should read the abstract with no prior discussion context.

In [7]:
print("Generating round 1 independent reviews...\n")

round1_reviewers = [methodology_reviewer, domain_reviewer, skeptical_reviewer]

# Each reviewer speaks independently — no history yet
seed_opinions   = [reviewer.speak() for reviewer in round1_reviewers]
seed_usernames  = [reviewer.get_actor_name()    for reviewer in round1_reviewers]

for name, opinion in zip(seed_usernames, seed_opinions):
    print(f"{'─' * 60}")
    print(f"[{name}]")
    print(opinion)

print("─" * 60)
print("Round 1 complete.")

Generating round 1 independent reviews...

────────────────────────────────────────────────────────────
[MethodologyReviewer]
**Review by Methodology Reviewer**

**Title:** Attention Is All You Need

**Abstract Summary:**
The authors present a novel approach to sequence transduction models, proposing a Transformer architecture that relies solely on attention mechanisms, eliminating the need for recurrence and convolutions. The paper reports superior performance on machine translation tasks compared to existing models, along with enhanced parallelizability and reduced training time.

**Strengths:**
1. **Innovative Architecture:** The Transformer model introduces a fresh perspective by focusing exclusively on attention mechanisms, which could potentially lead to more efficient and scalable models.
2. **Performance Claims:** The experimental results demonstrate that the proposed model outperforms existing methods on machine translation tasks, suggesting significant advancements in the fie

## 6. Round 2 and Round 3 — Discussion and meta-review

Now we hand over to SynDisco's `Discussion` class.

- The three reviewers are the participants
- The round-1 outputs become **seed opinions**, so every agent starts with the full prior context
- We run **4 prompted turns**: 3 for the reviewers to update their critiques (round 2), then 1 for the meta-reviewer (round 3)
- A custom turn order puts the meta-reviewer last

### Turn order design

  
We provide `[methodology, domain, skeptical, meta]` and set `conv_len=4`, which gives us exactly one full cycle.

In [ ]:
from syndisco import Discussion, RoundRobin

# All four agents are participants; the turn order encodes the round structure
all_agents = [
    methodology_reviewer,
    domain_reviewer,
    skeptical_reviewer,
    meta_reviewer,
]

# RoundRobin cycles through actors in the order they are given.
turn_manager = RoundRobin(actors=all_agents) 

discussion = Discussion(
    next_turn_manager=turn_manager,
    users=all_agents,
    # Round-1 reviews are injected as seeds so every agent sees them as context
    seed_opinions=seed_opinions,
    seed_opinion_usernames=seed_usernames,
    # Each agent sees the last 6 messages as context (enough for all seeds + some turns)
    history_context_len=6,
    # 3 updated critiques (round 2) + 1 meta-review (round 3)
    conv_len=4,
)

print("Discussion object created.")
print(f"Seed opinions : {len(seed_opinions)}")
print(f"Prompted turns: {discussion.conv_len}")

In [ ]:
print("Running rounds 2 and 3...\n")
discussion.begin(verbose=True)

## 7. Inspecting and saving the logs

`Discussion.get_logs()` returns a `Logs` object that you can iterate, index, and export to JSON.

In [ ]:
logs = discussion.get_logs()

print(f"Total log entries: {len(logs)}\n")
print(f"{'Entry':<6} {'Speaker':<22} {'Text preview'}")
print("-" * 80)
for i, entry in enumerate(logs):
    preview = entry['text'][0].replace('\n', ' ')
    print(f"{i:<6} {entry['name']:<22} {preview}...")

## 8. Side-by-side comparison

Let's print the baseline single-prompt review alongside the meta-reviewer's final judgment.

In [ ]:
# The meta-reviewer is always the last entry in the log
meta_review_entry = logs[-1]

divider = "=" * 60

print(divider)
print("BASELINE — single-prompt review")
print(divider)
print(baseline_review)

print()
print(divider)
print(f"MULTI-AGENT — {meta_review_entry['name']} final judgment")
print(divider)
print(meta_review_entry['text'])

## 9. Discussion and reflection

Consider the following questions with your group:

1. **Specificity** — Does the multi-agent review identify more concrete weaknesses than the single-prompt review? Why might that be?

2. **Perspective coverage** — Which aspects of the paper did each reviewer catch that the others missed? Would a single prompt have covered all of them?

3. **Iteration effect** — Compare the round-2 critiques against the round-1 seeds. Did the reviewers update their positions after reading each other? In what direction?

4. **Institutional simulation** — Real peer review is also a multi-agent, multi-round process. What does this exercise tell us about when synthetic discussion is a good model of a real process?

5. **Failure modes** — When would this system *not* work well? Think about: agent agreement bias ("sycophancy cascades"), the quality ceiling imposed by a shared model, and the absence of genuine domain knowledge.

---

## Summary

In this workshop you:

1. Observed that a **single-prompt** review tends to be general and surface-level
2. Built a **role-specialised multi-agent** system where each agent has a focused mandate
3. Used **seed opinions** to inject round-1 independent reviews as shared context
4. Let SynDisco's **`Discussion`** class manage the turn order and context window across rounds 2 and 3
5. Produced a **structured meta-review** that synthesises all perspectives

The key insight is that **structured reasoning emerges from interaction**. Specialised agents constrained to their own perspective surface different aspects of the same paper, and the meta-reviewer can only produce a grounded synthesis because the prior turns have already done the analytical legwork. This mirrors how real peer review works — and explains why it is hard to replicate with a single prompt.